# Session shoreline representative + curve boxplot (Application II helper)

This notebook builds **distance-to-baseline shoreline curves** for a single session (folder of rectified shoreline CSVs), computes a **functional depth** ordering (Fraiman–Muniz style), produces a **curve boxplot summary**, and exports a **representative shoreline CSV** that you can feed into `build_beach_surface.ipynb`.

**Inputs**
- `BASELINE_CSV`: baseline polyline in world/rectified coordinates (meters).
- `SESSION_DIR`: folder containing many rectified shoreline CSVs for one session (or one clip).

**Outputs**
- `rep_shoreline_world.csv`: representative shoreline polyline in world coords (2D).
- `curve_boxplot_summary.csv`: median/central-band curves in distance space.
- `curve_boxplot.png`: diagnostic plot.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# CONFIG (edit these)
# -----------------------------
BASE_DIR = Path('./csv/seabright_new_rec/')
BASELINE_CSV = BASE_DIR / 'baseline' / 'baseline_warped.csv'

SESSION_GLOB = '*_warped.csv'
OUT_ROOT = Path('./csv/seabright_new_rec_averaged2/')
OUT_ROOT.mkdir(parents=True, exist_ok=True)

N_BASELINE_SAMPLES = 400

MIN_HIT_FRAC_PER_S = 0.7
DROP_CURVES_WITH_NANS = True

# Build the list of session folders (each folder = one session)
session_dirs = sorted([
    p for p in BASE_DIR.iterdir()
    if p.is_dir() and p.name != 'baseline'
])

print(f"Found {len(session_dirs)} session folders.")


Found 12 session folders.


In [ ]:
def read_polyline_csv(csv_path: Path, prefer_world=True):
    """
    Load a polyline from your CSV format.
    - If world coords exist, uses X_warped/Y_warped (preferred).
    - Otherwise falls back to x/y.

    Returns: (N,2) ndarray in the chosen coordinate system.
    """
    df = pd.read_csv(csv_path)

    # If there's a per-vertex ordering column, respect it
    if 'vertex_index' in df.columns:
        df = df.sort_values('vertex_index')

    if prefer_world and {'X_warped','Y_warped'}.issubset(df.columns):
        pts = df[['X_warped','Y_warped']].to_numpy(float)
    else:
        pts = df[['x','y']].to_numpy(float)

    # Drop NaNs
    pts = pts[~np.isnan(pts).any(axis=1)]
    return pts


def resample_polyline(pts: np.ndarray, n_samples: int):
    """Resample a polyline to n_samples by arc-length parameterization."""
    pts = np.asarray(pts, float)
    if len(pts) < 2:
        raise ValueError('Polyline must have at least 2 points')

    seg = np.diff(pts, axis=0)
    seg_len = np.hypot(seg[:, 0], seg[:, 1])
    s = np.concatenate([[0.0], np.cumsum(seg_len)])
    total = s[-1]
    if total == 0:
        return np.repeat(pts[:1], n_samples, axis=0)

    s_new = np.linspace(0, total, n_samples)
    x_new = np.interp(s_new, s, pts[:, 0])
    y_new = np.interp(s_new, s, pts[:, 1])
    return np.column_stack([x_new, y_new])


def compute_normals(baseline_pts: np.ndarray):
    """Compute unit normals for a polyline baseline."""
    b = np.asarray(baseline_pts, float)
    # Tangents via central differences
    t = np.zeros_like(b)
    t[1:-1] = b[2:] - b[:-2]
    t[0] = b[1] - b[0]
    t[-1] = b[-1] - b[-2]

    # Normalize tangents
    t_norm = np.linalg.norm(t, axis=1, keepdims=True)
    t_norm[t_norm == 0] = 1.0
    t = t / t_norm

    # Rotate tangent by +90 deg to get a normal
    n = np.column_stack([-t[:, 1], t[:, 0]])

    # Normalize normals
    n_norm = np.linalg.norm(n, axis=1, keepdims=True)
    n_norm[n_norm == 0] = 1.0
    n = n / n_norm
    return n


In [ ]:
def intersect_ray_with_segment(o, d, a, b, eps=1e-9):
    """
    Line: p(t) = o + t d, t in (-inf, inf)
    Segment: q(u) = a + u (b-a), u in [0,1]
    Returns t if intersection exists, else None.
    """
    o = np.asarray(o, dtype=float)
    d = np.asarray(d, dtype=float)
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    v = b - a
    M = np.column_stack((d, -v))
    det = np.linalg.det(M)
    if abs(det) < eps:
        return None

    t, u = np.linalg.solve(M, a - o)
    if -eps <= u <= 1 + eps:
        return t
    return None



def build_distance_curve(shore_pts: np.ndarray, baseline_pts: np.ndarray, normals: np.ndarray):
    """
    For each baseline station i, cast a ray from baseline_pts[i] along normals[i]
    and record the nearest intersection distance with the shoreline polyline.

    Returns: d (M,) where M=len(baseline_pts), NaN if no intersection.
    """
    shore = np.asarray(shore_pts, float)
    if len(shore) < 2:
        return np.full(len(baseline_pts), np.nan)

    dvals = np.full(len(baseline_pts), np.nan, dtype=float)

    for i, (o, dvec) in enumerate(zip(baseline_pts, normals)):
        hits = []
        for j in range(len(shore) - 1):
            t = intersect_ray_with_segment(o, dvec, shore[j], shore[j + 1])
            if t is not None:
                hits.append(t)
        if hits:
            # choose the intersection closest to the baseline point along the transect line
            t_best = min(hits, key=lambda x: abs(x))
            dvals[i] = float(abs(t_best))   # store unsigned distance for reconstruction


    return dvals


In [ ]:
def functional_depth_rank(curves: np.ndarray):
    """
    Fraiman–Muniz style depth.
    curves: (N, M) -> N curves, M points alongshore
    Returns depth array of length N (higher = more central).
    """
    N, M = curves.shape
    depths = np.zeros(N, dtype=float)

    # handle NaNs conservatively (should be filtered before this, but just in case)
    curves = np.where(np.isnan(curves), np.nanmedian(curves, axis=0, keepdims=True), curves)

    for j in range(M):
        col = curves[:, j]
        order = np.argsort(col)
        ranks = np.empty(N, dtype=int)
        ranks[order] = np.arange(N) + 1  # 1..N

        center = (N + 1) / 2.0
        denom = (N - 1) / 2.0 if N > 1 else 1.0
        U = np.abs(ranks - center) / denom  # [0, 1]
        depths += (1.0 - U)

    depths /= M
    return depths


def build_curve_boxplot(curves_valid: np.ndarray):
    """
    Returns median curve (deepest observed curve), central 50% envelope, and pointwise-median curve.
    curves_valid: (N,M) with no NaNs
    """
    N, M = curves_valid.shape
    depths = functional_depth_rank(curves_valid)
    order = np.argsort(-depths)

    curves_sorted = curves_valid[order]
    median_observed = curves_sorted[0]

    # pointwise median across all curves (often better for reconstruction stability)
    median_pointwise = np.median(curves_valid, axis=0)

    # central 50% envelope (deepest half)
    n_central = max(1, int(np.ceil(0.5 * N)))
    central = curves_sorted[:n_central]
    lower = np.min(central, axis=0)
    upper = np.max(central, axis=0)

    return {
        'depths': depths,
        'order': order,
        'median_observed': median_observed,
        'median_pointwise': median_pointwise,
        'lower': lower,
        'upper': upper,
    }


In [ ]:
def process_session(SESSION_DIR: Path, OUT_DIR: Path):
    """Process a single session folder: build curves, compute boxplot, export representative shoreline."""
    
    # Collect shoreline CSVs for this session
    shore_csvs = sorted([p for p in SESSION_DIR.glob(SESSION_GLOB)
                         if p.name != BASELINE_CSV.name])

    if not shore_csvs:
        print(f"  [skip] No shoreline CSVs")
        return

    print(f"  Found {len(shore_csvs)} shoreline CSVs")
    
    # Load + resample baseline
    baseline_raw = read_polyline_csv(BASELINE_CSV, prefer_world=True)
    baseline_pts = resample_polyline(baseline_raw, N_BASELINE_SAMPLES)
    normals = compute_normals(baseline_pts)
    u = np.linspace(0.0, 1.0, N_BASELINE_SAMPLES)
    
    # Build distance curves d(u) for each shoreline
    curves = []
    valid_names = []

    for csv_path in shore_csvs:
        shore_pts = read_polyline_csv(csv_path, prefer_world=True)
        d = build_distance_curve(shore_pts, baseline_pts, normals)
        curves.append(d)
        valid_names.append(csv_path.name)

    curves = np.vstack(curves)  # (N_curves, M)
    
    # Mask alongshore stations where enough curves have valid intersections
    hit_frac = np.mean(~np.isnan(curves), axis=0)
    mask = hit_frac >= MIN_HIT_FRAC_PER_S
    print(f"  Valid alongshore stations: {int(mask.sum())}/{len(mask)}")

    curves_valid = curves[:, mask]

    if DROP_CURVES_WITH_NANS:
        keep = ~np.isnan(curves_valid).any(axis=1)
        curves_valid = curves_valid[keep]
        kept_names = [n for n,k in zip(valid_names, keep) if k]
    else:
        kept_names = valid_names

    print(f"  Curves after NaN filtering: {curves_valid.shape[0]}")
    
    if curves_valid.shape[0] < 3:
        print(f"  [skip] Not enough valid curves (<3)")
        return
    
    # Compute curve boxplot summary
    summary = build_curve_boxplot(curves_valid)
    median_obs = summary['median_observed']
    median_pw  = summary['median_pointwise']
    lower = summary['lower']
    upper = summary['upper']
    u_valid = u[mask]
    
    # Back-project representative shoreline to world coordinates
    REP_CHOICE = 'median_pointwise'
    rep_d = median_pw if REP_CHOICE == 'median_pointwise' else median_obs

    baseline_valid = baseline_pts[mask]
    normals_valid = normals[mask]
    rep_xy = baseline_valid + rep_d[:, None] * normals_valid

    rep_df = pd.DataFrame({
        'u': u_valid,
        'X_warped': rep_xy[:,0],
        'Y_warped': rep_xy[:,1],
        'rep_choice': REP_CHOICE,
    })

    rep_csv = OUT_DIR / 'rep_shoreline_world.csv'
    rep_df.to_csv(rep_csv, index=False)
    print(f"  ✓ Wrote {rep_csv.name}")
    
    # Export curve-boxplot summary
    box_df = pd.DataFrame({
        'u': u_valid,
        'median_observed': median_obs,
        'median_pointwise': median_pw,
        'lower_central50': lower,
        'upper_central50': upper,
        'hit_frac': hit_frac[mask],
    })

    box_csv = OUT_DIR / 'curve_boxplot_summary.csv'
    box_df.to_csv(box_csv, index=False)
    print(f"  ✓ Wrote {box_csv.name}")
    
    # Plot diagnostics
    plt.figure(figsize=(10,4))

    # plot a subset of curves for visibility
    step = max(1, curves_valid.shape[0] // 50)
    for c in curves_valid[::step]:
        plt.plot(u_valid, c, alpha=0.15)

    plt.fill_between(u_valid, lower, upper, alpha=0.25)
    plt.plot(u_valid, median_pw, linewidth=2, label='median (pointwise)')
    plt.plot(u_valid, median_obs, linewidth=2, label='median (deepest observed)')

    plt.xlabel('alongshore parameter u')
    plt.ylabel('distance to baseline (meters, in rectified coords)')
    plt.title(f'{SESSION_DIR.name}: curve boxplot (central 50% + representative)')
    plt.legend()
    plt.tight_layout()

    out_png = OUT_DIR / 'curve_boxplot.png'
    plt.savefig(out_png, dpi=200)
    plt.close()
    print(f"  ✓ Wrote {out_png.name}")


# Run processing for all session folders
for SESSION_DIR in session_dirs:
    OUT_DIR = OUT_ROOT / SESSION_DIR.name
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    print(f"\n[Processing] {SESSION_DIR.name}")
    process_session(SESSION_DIR, OUT_DIR)

print("\n✅ All sessions processed!")


In [ ]:
# import matplotlib.pyplot as plt

# s1 = pd.read_csv(shoreline_files[0])

# plt.figure()
# plt.plot(b["X_warped"], b["Y_warped"], "-o", label="baseline")
# plt.plot(s1["X_warped"], s1["Y_warped"], "-", label="shoreline example")
# plt.axis("equal")
# plt.legend()
# plt.title("Baseline vs one shoreline (world coords)")
# plt.show()


: 

In [1]:
# import numpy as np
# import pandas as pd

# b = pd.read_csv(BASELINE_CSV)
# s = pd.read_csv(shoreline_files[0])

# p0 = b[["X_warped","Y_warped"]].to_numpy()[0]
# p1 = b[["X_warped","Y_warped"]].to_numpy()[-1]
# t = p1 - p0
# t = t / np.linalg.norm(t)

# # candidate normals
# n1 = np.array([-t[1], t[0]])
# n2 = -n1

# # take a shoreline point near the middle
# q = s[["X_warped","Y_warped"]].to_numpy()[len(s)//2]

# print("dot with n1:", np.dot(q - p0, n1))
# print("dot with n2:", np.dot(q - p0, n2))
